In [ ]:
import ray
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import statsmodels.api as sm
from tqdm.notebook import tqdm
from SALib.analyze import pawn
import matplotlib.pyplot as plt
from adjustText import adjust_text
from SALib.sample.sobol import sample
from wellbeing_abm import WellbeingSim
from string import ascii_lowercase as alc
from matplotlib.colors import TwoSlopeNorm
from itertools import product, combinations
from matplotlib.ticker import FormatStrFormatter
from sklearn.preprocessing import KBinsDiscretizer
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from sklearn.feature_selection import mutual_info_regression

paper_rc = {'lines.linewidth': 2, 'lines.markersize':7, "text.usetex": True}                  
sns.set_context("paper", rc=paper_rc, font_scale=1.3)

from helper_functions import *
from wellbeing_abm import WellbeingSim

# **Default parameters and settings**

In [ ]:
N = 50 # N = 500 for paper results
reps = 2 # reps = 100 for paper results
samples = 16 # samples = 8192 for paper results
num_cpus = 7
run_lsa = True
run_gsa = True
run_traps = True
run_shock = True
run_stylised = True
run_easterlin = True


default_params = {
    # Network params
    'N': N, 'p': 0.5, 'do_network_update':False, 'network_type': 'sda', 
    'homophily': -1, 'avg_degree': 10, 'p_rewire': 0.1, 'wb_in_dist':False,

    # Simulation params
    'n_steps': 99,  'alpha': np.random.uniform(0, 0.5, size=N), 
    'soc_comp_w': np.random.uniform(0, 1, size=N), 'shock_sigma':0.005, 
    'soc_to_wb':False,  'adapt_kind': 'adapt_lvl', 'slope_sig':-0.0001,
    'beta':0, 'perception':'sigmoid',

    # Income params
    'wb_to_cap':np.random.uniform(0, 2, size=N), 'mu': 10, 'sigma': 0.1, 
    'capital_dist': 'lognormal', 'soc_comparison':True, 'comparison_kind':'quantile',
    'quantile':0.5, 'k':1, 'growth_rate':0.002,  
    
    # Event params
    'event_type':'fixed+random',
    'int_freq':50, 'int_size':-10_000, 'rel_size':0,
}

model = WellbeingSim(params=default_params)

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15,6))
colours = sns.color_palette("tab10")
values = [model.swb_setpoint, model.capital[:,0], model.alpha,
          model.soc_comp_w, model.wb_to_cap, model.network.sum(axis=1)]
titles = ['Set-point ($swb_{i(t=0)}$)', 'Initial Income ($y_{i(t=0)}$)', 
          'Adaptation ($\\alpha_i$)', 'Comparison ($w_i$)', 
          'Feedback ($\\delta_i$)', 'Degree ($k_i$)']
for i, ax in enumerate(axes.flatten()):
    ax.set_xlabel(titles[i])
    if i==1: log_scale=False
    else: log_scale=False
    sns.histplot(values[i], ax=ax, color=colours[i], log_scale=log_scale)
    ax.set(ylabel=None)
plt.suptitle(f'Initial Values (N={N})')
plt.tight_layout()

In [ ]:
params = default_params.copy()
params['N'] = 3
params['p'] = [1,0,1]
params['comparison_kind'] = 'quantile'
# params['k'] = 2
params['shock_sigma'] = 0
params['wb_to_cap'] = 0
params['soc_comp_w'] = np.array([0.5, 0.5, 0.5])
params['alpha'] = np.array([0.25, 0.25, 0.25])

model = WellbeingSim(params=params)
network = np.array([[0, 1, 1], [1, 0, 0], [1, 0, 0]])
model.network = nx.to_scipy_sparse_array(nx.from_numpy_array(network), dtype=bool)
model.wellbeing[:,0] = model.wellbeing[:,1] = model.swb_setpoint = np.array([0.5, 0.55, 0.45])
model.capital[:,0] = model.capital[:,1] = model.al_capital[:,0] = model.al_capital[:,1] = np.array([20_000, 15_000, 25_000])
model.run_simulation()

capital = model.capital
wellbeing = model.wellbeing
al_capital = model.al_capital

colours = ['coral', 'steelblue', 'green']
styles = ['-', '--', ':']
markers = [None, 'o', 'X']
fig, ax = plt.subplots(figsize=(12,3))
ax_twin = ax.twinx()
for agent in range(3):
    sns.lineplot(wellbeing[agent,:], color=colours[agent], ax=ax_twin,
                 label='swb$_{' + f'{agent+1}' + 't}$', linestyle=styles[agent], marker=markers[agent])
    sns.lineplot(capital[agent,:], color='grey', linestyle=styles[agent], marker=markers[agent], alpha=0.5,
                label='y$_{' + f'{agent+1}' + 't}$', ax=ax)
    # sns.lineplot(al_capital[agent,:], color=colours[agent], linestyle=':', marker='X',
    #             label='AL$_{' + f'{agent}' + 't}$')

# _ = plt.legend(ncol=3)
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
ax_twin.legend(lines + lines2, labels + labels2, loc='upper right', ncol=2).set_zorder(103)
ax.get_legend().remove()
ax.set_xlabel('Time')
ax.set_ylabel('Income')
ax_twin.set_ylabel('SWB')
ax_twin.set_ylim(0,1)
ax.set_title('SWB and income of three agents over time')

# **How shocks to income are perceived (i.e. affect SWB)**

In [ ]:
ls = ['-','--',':','-.']
ms = ['o', 'x']
cs = sns.color_palette("tab10")
plt.figure(figsize=(6,5))
for i, current_swb in enumerate([0.2, 0.5, 0.7]):
    x0_point = current_swb / (1-current_swb)
    diff_cap = np.linspace(-20_000, 20_000, 25)

    slope, lamb = -0.0001, 2.25
    swb = np.where(diff_cap<0, 
                   x0_point / (x0_point + np.exp(slope*lamb*diff_cap)),
                   x0_point / (x0_point + np.exp(slope*diff_cap))
                  )
    ax = sns.lineplot(x=diff_cap, y=swb, label='$swb_{i(t=0)}=' + f'{current_swb}$', 
                      linestyle=ls[i], color=cs[i])
    # Concave version
    r = -20_000
    swb_conc = np.where(diff_cap<0, 
                   sqrt_concave(x=diff_cap, y0=current_swb, r=r),
                   sqrt_concave(x=diff_cap, y0=current_swb, r=r * lamb)
                  )
    ax = sns.lineplot(x=diff_cap, y=swb_conc, label='Conc. $swb_{i(t=0)}=' + f'{current_swb}$', ax=ax,
                      linestyle=ls[i], marker=ms[0], color=cs[i])
ax.set_xlabel('$y_{it} - AL_{it}$')
ax.set_ylabel('$swb_{it}$')
ax.set_axisbelow(True)
ax.grid(True)
_ = ax.set_title('Effect of income shocks on SWB')
plt.tight_layout()
plt.show()

ls = ['-','--',':','-.']
plt.figure(figsize=(6,5))
for i, feedback in enumerate([0, 0.5, 1, 2]):
    x = np.linspace(-1, 1, 100)
    y = 2 / (1 + np.exp(-feedback*(x)))
    ax = sns.lineplot(x=x, y=y, label='$\\delta_i=' + f'{feedback}$', linestyle=ls[i])
ax.set_xlabel('$swb_{i(t-1)} - swb_{i(t-2)}$')
ax.set_ylabel('Multiplier')
ax.set_axisbelow(True)
ax.grid(True)
_ = ax.set_title('Effect of $\\delta_i$ on the multiplier')
plt.tight_layout()

# **Individual dynamics**

In [ ]:
params = default_params.copy()
params['N'] = 1
params['p'] = 1
params['alpha'] = 0.15
params['soc_comp_w'] = 0
params['wb_to_cap'] = 1
params['soc_comparison'] = False
params['network_type'] = 'random'
params['capital_dist'] = 'homogeneous'

model = WellbeingSim(params=params)
model.wellbeing[:,0] = model.wellbeing[:,1] = model.swb_setpoint = 0.5
model.capital[:,0] = model.capital[:,1] = model.al_capital[:,0] = model.al_capital[:,1] = 20_000
model.run_simulation()

plt.figure(figsize=(10,3))
markers = ['o', '^', 'd', '>', '<']
labels = ['SWB', 'Income', 'AL(income)']
lines = [model.wellbeing[0], model.capital[0], model.al_capital[0]]

ax = plt.gca()
ax_twin = ax.twinx()
ax_twin._get_lines.get_next_color()
for i, line in enumerate(lines):
    if i==0: plot_ax=ax
    else: plot_ax=ax_twin
    sns.lineplot(x=range(len(line)), y=line, label=labels[i], ax=plot_ax, marker=markers[i])

lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
ax.legend(lines + lines2, labels + labels2)
ax_twin.get_legend().remove()
plt.xlabel('Time')
ax.set_ylabel('SWB')
ax_twin.set_ylabel('Income')
ax.set_title('SWB, income and adaptation level for an individual over time')
plt.tight_layout()

# **Stylised facts**

In [ ]:
if run_stylised:
    hom_names = {-1:'Random',2:'Segregated'}
    for w_upper, homophily, mu, sigma in [(1, -1, 10, 0.1), (1, 2, 10, 0.1), (1, -1, 10, 0.8), (1, 2, 10, 0.8)]:
    
        # Run the model with updated parameters and calculate outputs0
        params = default_params.copy()
        N = params['N']
        params.update({
            'N':N, 'mu':mu, 'sigma':sigma, 'homophily':homophily, 'p':0.5, 'quantile':0.5, 'growth_rate':0.002,
            'rel_size':0, 'soc_comparison':True, 'int_size':-10_000, 'n_steps':99,
            'wb_to_cap':np.random.uniform(0, 2, size=N),
            'soc_comp_w':np.random.uniform(0, w_upper, size=N), 
            'alpha':np.random.uniform(0, 0.5, size=N),
        })
        # Run the model
        model = WellbeingSim(params=params)
        model.run_simulation()
        plot_vars, cap, swb, shocked_idx = calculate_stats(model)

        welfare_cap = [calc_welfare(cap[:,t]) for t in range(cap.shape[1])]
        welfare_swb = [calc_welfare(swb[:,t]) for t in range(swb.shape[1])]
        
        ls = ['-','--',':','-.']
        fig, ax = plt.subplots(figsize=(12,3))
    
        sns.lineplot(data=pd.DataFrame(cap).melt(), y='value', x='variable', 
                     label='Avg. Income', linestyle=ls[0], errorbar=("pi", 90))
        ax_twin = ax.twinx()
        ax_twin._get_lines.get_next_color()
        sns.lineplot(data=pd.DataFrame(swb).melt(), y='value', x='variable', 
                     label='Avg. SWB', linestyle=ls[0], errorbar=("pi", 90), ax=ax_twin, marker='o')
        correlations = [pd.DataFrame([cap[:,t], swb[:,t]]).T.corr(method='spearman').iloc[0,1] for t in range(model.n_steps+1)]
        sns.lineplot(data=correlations, label='Corr(Inc,SWB)', linestyle=ls[2], ax=ax_twin, marker='D')
        # sns.lineplot(data=welfare_cap, label='Welfare(Inc)', linestyle=ls[1], color='steelblue', ax=ax)
        sns.lineplot(data=welfare_swb, label='Welfare(SWB)', linestyle=ls[1], color='coral', ax=ax_twin, marker='X')

        # Calculate regression slopes
        t = np.array(list(range(swb.shape[1])))
        mean_swb = swb.mean(axis=0)
        mean_cap = cap.mean(axis=0)
        const_swb, slope_swb = sm.OLS(mean_swb, sm.add_constant(t), missing='drop').fit().params
        const_cap, slope_cap = sm.OLS(mean_cap, sm.add_constant(t), missing='drop').fit().params
        # sns.lineplot(x=t, y=slope_cap*t + const_cap, ax=ax, label=None, linestyle='-', lw=2, color='black')
        # sns.lineplot(x=t, y=slope_swb*t + const_swb, ax=ax_twin, label=None, linestyle='--', lw=2, color='black')
        print("Slope Inc: ", slope_cap / 1000)
        print("Slope SWB: ", slope_swb)
        
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax_twin.get_legend_handles_labels()
        ax_twin.legend(lines + lines2, labels + labels2, loc='upper center', ncol=5).set_zorder(103)
        ax.get_legend().remove()
        ax.set_title(f'Income, SWB and their cross-sectional Spearman correlation with {hom_names[homophily].lower()} comparisons')
        ax.set_xlabel('Time')
        ax.set_ylabel('Income')
        ax_twin.set_ylabel('SWB / Correlation')
        ax_twin.set_ylim(0,1)
        ax.annotate('a)', xy=(0.01,0.9), xycoords='axes fraction', fontsize=20)
        plt.show()
        
        title = f'{hom_names[homophily]} comparison groups with ' +\
            '$y_{i(t=0)} \\sim LN(\\mu=' + f'{mu},\\sigma={sigma}) + 1$' + ' and ' + '$w_{max}' + f'={w_upper}$'
        fig, axes = plt.subplots(ncols=3, nrows=1, figsize=(14,4), sharey=False)
        plt.suptitle(title)
        annotations = ['a)', 'b)', 'c)', 'd)', 'e)', 'f)']

        for k in ['inc', 'swb']:
            print(f'Initial Welfare {k} :', np.round(plot_vars[f'Welfare ({k}, $t=0$)'].mean(), 2), 
                  'final welfare:', np.round(plot_vars[f'Welfare ({k})'].mean(), 2),
                  'difference:', np.round(plot_vars[f'$\\Delta$Welfare ({k})'].mean(), 2))
            print()
    
        combos = [
            ('Final Income ($y_{i(t=99)}$)', 'Final SWB ($swb_{i(t=99)}$)', 'Comparison ($w_i$)', 0, 1),
            ('$\\Delta y_i = y_{i(t=99)} - y_{i(t=0)}$', '$\\Delta swb_i = swb_{i(t=99)} - swb_{i(t=0)}$', 'Comparison ($w_i$)', 0, 1),
            ('Initial Income ($y_{i(t=0)}$)', 'Recovery Time', 'Comparison ($w_i$)', 0, 1)
        ]
        for i, (x, y, hue, vmin, vmax) in enumerate(combos):
            
            ax = axes[i]
            ax = scatterplot(plot_vars, x, y, hue, vmin=vmin, vmax=vmax, vcenter=(vmin+vmax)/2, cmap='viridis', ax=ax, fig=fig)
            ax.annotate(annotations[i+1], xy=(0.01,0.9), xycoords='axes fraction', fontsize=20)
            # if i!=1: ax.set_xscale("log")
            xlims, ylims = ax.get_xlim(), ax.get_ylim()
            # Add regr. line
            if i<1:
    
                # Estimate regression line
                if i==0: 
                    x_t = np.log(plot_vars[x])
                else: 
                    x_t = plot_vars[x] / 1000
                reg_model = sm.OLS(plot_vars[y], sm.add_constant(x_t), missing='drop')
                reg_results = reg_model.fit()
                const, slope = reg_results.params
    
                # Configure line title
                if i==0: 
                    line_title = '$swb_{i(t=99)} = ' + str(round(slope, 2)) + 'ln(y_{i(t=99)})' + str(round(const, 2)) + '$'
                else: 
                    line_title = '$\\Delta swb_i = ' + str(round(slope, 3)) + '\\Delta y_i + ' + str(round(const, 2)) + '$'
    
                ax.set_title(line_title)
                sns.lineplot(x=plot_vars[x], y=slope*x_t + const, ax=ax, label=None, linestyle='-', lw=2, color='black', zorder=-1)
                ax.set_xlim(xlims[0], xlims[1])
                ax.set_ylim(ylims[0], ylims[1])
            
    
        plt.tight_layout()
        plt.show()
        
        # Part 2
        fig, axes = plt.subplots(ncols=3, nrows=1, figsize=(14,4), sharey=False)
        plt.suptitle(title)

        combos = [
            ('Comparison ($w_i$)', 'Long-term Change', 'Net change in income', 'redblue'),
            ('Initial Income ($y_{i(t=0)}$)', 'Instability (CV)', 'Comparison ($w_i$)', 'viridis'),
            ('Feedback ($\\delta_i$)', 'Long-term Change', 'Net change in income', 'redblue')
        ]
        if homophily==-1:
            annotations = ['a)', 'b)', 'c)']
        else:
            annotations = ['d)', 'e)', 'f)']
        for i, (x, y, hue, cmap) in enumerate(combos):
            
            ax = axes[i]
            if cmap=='redblue':
                vcenter = 0
                vmin = plot_vars[hue].min()
                vmax = plot_vars[hue].max()
            else:
                vcenter, vmin, vmax = 0.5, 0, 1
                # ax.set_xscale("log")
            ax = scatterplot(plot_vars, x, y, hue, vmin=vmin, vmax=vmax, vcenter=vcenter, cmap=cmap, ax=ax, fig=fig)
            ax.annotate(annotations[i], xy=(0.01,0.9), xycoords='axes fraction', fontsize=20)
    
            if i==0: ax.axhline(y=plot_vars[y].mean(), linewidth=3, color='black', linestyle='dashed')
        
        plt.tight_layout()

# **Comparison groups**

In [ ]:
for homophily in [-1, 2]:
    runs = []
    for rep in range(reps):
        params = default_params.copy()
        params['homophily'] = homophily
        params['sigma'] = 0.8
        params['q'] = 0.5
        params['growth_rate'] = 0.002
        # params['wb_in_dist'] = True
        # params['do_network_update'] = True
        params['alpha'] = np.random.uniform(0, 0.5, size=N)
        params['wb_to_cap'] = np.random.uniform(0, 2, size=N)
        params['soc_comp_w'] = np.random.uniform(0, 1, size=N)
        runs += [params]
    
    obj_ids = [comparison_parallel.remote(run) for run in runs]
    output_results = [ray.get(i) for i in tqdm(obj_ids)]
    all_frames = pd.concat(output_results, keys=[i for i in range(len(output_results))])
    
    fig, ax = plt.subplots(figsize=(10,3))
    choices = ['Downward', 'Mixed', 'Upward']
    sns.lineplot(data=all_frames, x='Time', y='SWB', hue='Income', markers=True,
                 style='Comparison', ax=ax, errorbar=None,#('pi',90),
                 hue_order=['Low (<50%)', 'High (>=50%)'], style_order=choices)
    if homophily==-1: comp_groups='random'
    else: comp_groups='segregated'
    _ = ax.set_title(f'SWB of different ({comp_groups}) comparison and income groups')
    if params['homophily']==-1:
        ax.annotate('a)', xy=(0.01,0.88), xycoords='axes fraction', fontsize=20)
    elif params['homophily']==2:
        ax.annotate('b)', xy=(0.01,0.88), xycoords='axes fraction', fontsize=20)
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    # sns.lineplot([gini_coefficient(model.capital[:,t]) for t in range(model.n_steps+1)], label='Gini')
    plt.tight_layout()
    plt.show()

In [ ]:
ray.shutdown()
assert not ray.is_initialized()

# **Create output column names**

In [ ]:
# Rename parameters for plots
rename = {
    'alpha':'Adaptation',
    'wb_to_cap':'Feedback',
    'shock_sigma':'$\\sigma_{shock}$',
    'soc_comp_w': '$w_{comp}$',
    'homophily':'Homophily',
    'avg_degree':'Degree',
    'sigma':'Inequality',
    'comparison_kind':'Comparison',
    'do_network_update':'Rewiring',
    'quantile':'Quantile',
    'alpha_lower':'$\\alpha_{min}$',
    'w_lower':'$w_{min}$',
    'alpha_upper':'$\\alpha_{max}$',
    'w_upper':'$w_{max}$',
    'fb_upper':'$\\delta_{max}$',
    'growth_rate':'Growth Rate ($g$)',
    'beta':'$\\beta$',
}

output_cols = ['$\\frac{\\beta_{swb}}{\\beta_{cap}}$', 
               '$\\Delta Welfare$ (swb)', '$\\Delta Welfare$ (cap)',
              '$\\Delta Welfare_{t=0}$ (swb)', '$\\Delta Welfare_{t=0}$ (cap)']

for metric in [' Avg. ', ' Std. ']:
    for group in ['', '(Below Median)', '(Shocked)']:
        for output in ['Long-term Change', 'Instability', 'Recovery Time', 'Sum Abs. Diffs.']:
            output_cols.append(output + metric + group)

np.array(output_cols)

# **Global Sensitivity Analysis**

In [ ]:
params = default_params.copy()
var_bounds = {
    'sigma':(0.1,1),
    'homophily':(0.2,2),
    'wb_to_cap':(0,3),
    'soc_comp_w':(0,1),
    'alpha':(0,0.5),
    'growth_rate':(0,0.01),
    'quantile':(0,1),
    'shock_sigma':(0,0.01),
    'avg_degree':(2,10),
    'Dummy':(0,1),
}
[*var_params],[*bounds] = zip(*var_bounds.items())
problem = {
        'num_vars': len(var_params),
        'names': var_params,
        'bounds': bounds
    }
param_values = sample(problem, samples)
params_gsa = pd.DataFrame(param_values, columns=var_params)

var_cols = params_gsa.columns
for param, value in params.items():

    if param not in var_cols:
        if isinstance(value, np.ndarray):
            params_gsa[param] = [value]*len(params_gsa)
        else:
            params_gsa[param] = value

# Every run should have new random values 
params_gsa['w_upper'] = params_gsa['soc_comp_w']
params_gsa['fb_upper'] = params_gsa['wb_to_cap']
params_gsa['alpha_upper'] = params_gsa['alpha']
params_gsa['alpha'] = params_gsa['alpha'].astype(object)
params_gsa['wb_to_cap'] = params_gsa['wb_to_cap'].astype(object)
params_gsa['soc_comp_w'] = params_gsa['soc_comp_w'].astype(object)

for row_id in range(params_gsa.shape[0]):
    params_gsa.at[row_id, 'alpha'] = np.random.uniform(0, params_gsa.at[row_id, 'alpha'], size=N)
    params_gsa.at[row_id, 'wb_to_cap'] = np.random.uniform(0, params_gsa.at[row_id, 'wb_to_cap'], size=N)
    params_gsa.at[row_id, 'soc_comp_w'] = np.random.uniform(0, params_gsa.at[row_id, 'soc_comp_w'], size=N)
params_gsa

In [ ]:
# Check if one run finishes without errors
if run_gsa:
    if not ray.is_initialized():
        ray.init(num_cpus=num_cpus)
    object_ref = run_model.remote(params_gsa.iloc[0])
    result = ray.get(object_ref)
    print(result)

In [ ]:
if run_gsa:
    runs = [row for i, row in params_gsa.iterrows()]
    obj_ids = [run_model.remote(run) for run in runs]
    params_gsa[output_cols] = [ray.get(i) for i in tqdm(obj_ids)]
    filename = 'data/gsa-data.json'
    with open('data/salib-problem.pickle', 'wb') as handle:
        pickle.dump(problem, handle, protocol=pickle.HIGHEST_PROTOCOL)
    with open('data/salib-parvalues.pickle', 'wb') as handle:
        pickle.dump(param_values, handle, protocol=pickle.HIGHEST_PROTOCOL)
    params_gsa.to_json(filename)

In [ ]:
if run_gsa:
    ray.shutdown()
    assert not ray.is_initialized()

In [ ]:
filename = 'data/gsa-data.json'
params_gsa = pd.read_json(filename)
with open('data/salib-problem.pickle', 'rb') as handle:
    problem = pickle.load(handle)
with open('data/salib-parvalues.pickle', 'rb') as handle:
    param_values = pickle.load(handle)

In [ ]:
for output_name in np.array(output_cols)[[1,5,6,7,17]]:
    
    output = params_gsa[output_name]
    xaxis = output_name
    
    # Calculate PAWN indices
    Si = pawn.analyze(problem, param_values, output, S=25)
    pawn_df = pd.DataFrame({
        'Parameter': problem['names'],
        'First Order': Si["mean"],
        'CV': Si["CV"],
        'Mean': Si["mean"]
    })
    pawn_df['Stdev'] = pawn_df['CV'] * pawn_df["Mean"]
    pawn_df.replace(rename, inplace=True)

    # Plot the output
    fig, [ax1, ax2] = plt.subplots(nrows=1, ncols=2, dpi=200,
                                   figsize=(15,5), width_ratios=[1,3])
    _ = sns.histplot(output, ax=ax1)
    ax1.set_title('GSA output measure', fontsize=16)
    ax1.set_xlabel(xaxis, fontsize=14)

    # Plot the PAWN indices
    ax2 = sns.barplot(x='Parameter', y='First Order', data=pawn_df, errorbar=None, ax=ax2, hue='Parameter') 
    _ = plt.errorbar(x=pawn_df['Parameter'], y=pawn_df['First Order'], 
                     yerr=pawn_df['Stdev'], fmt='none', c='black', capsize=5)
    title = 'PAWN Sensitivity Analysis'
    ax2.set_title(title, fontsize=16)
    ax2.set_xlabel('Parameter', fontsize=14)
    ax2.set_ylabel('Mean Pawn Index', fontsize=14)
    plt.xticks(rotation=20)

    # Rotate x-axis labels to be vertical and set font size
    # plt.xticks(ticks=range(len(params)), labels=pawn_df, rotation=0, fontsize=12)
#     plt.yticks(fontsize=12)
    plt.tight_layout()
    plot_path = '../plots/wellbeing-abm/'
    # plt.savefig(plot_path + 'pawn-' + output_name + '.pdf')
    plt.show()

# **Experiment with fraction of people shocked**

In [ ]:
# Create parameter combinations
params = default_params.copy()
inequality = [0.1, 0.8]
homophily = [-1, 2]
w_upper = [0.50, 1] 
alpha_upper = [0.10, 0.30, 0.50]
beta = [0]
growth_rate = [0, 0.002, 0.004]
shock_sigma = [params['shock_sigma']]
fb_upper = [2]
p = np.round(np.arange(0,1,0.1), 2)

permut = []
cols = ['sigma', 'homophily', 
        'soc_comp_w', 'alpha', 'beta',
        'p', 'growth_rate', 
        'shock_sigma', 'wb_to_cap']

for rep in range(reps):
    permut += list(product(inequality, homophily, 
                           w_upper, alpha_upper, beta,
                           p, growth_rate, 
                           shock_sigma, fb_upper))
params_sweep = pd.DataFrame(permut, columns=cols)

for param, value in params.items():

    if param not in cols:
        if isinstance(value, np.ndarray):
            params_sweep[param] = [value]*len(params_sweep)
        else:
            params_sweep[param] = value

# Every run should have new random values
params_sweep['w_upper'] = params_sweep['soc_comp_w']
params_sweep['fb_upper'] = params_sweep['wb_to_cap']
params_sweep['alpha_upper'] = params_sweep['alpha']
params_sweep['alpha'] = params_sweep['alpha'].astype(object)
params_sweep['wb_to_cap'] = params_sweep['wb_to_cap'].astype(object)
params_sweep['soc_comp_w'] = params_sweep['soc_comp_w'].astype(object)
for row_id in range(params_sweep.shape[0]):
    params_sweep.at[row_id, 'alpha'] = np.random.uniform(0, params_sweep.at[row_id, 'alpha'], size=N)
    params_sweep.at[row_id, 'wb_to_cap'] = np.random.uniform(0, params_sweep.at[row_id, 'wb_to_cap'], size=N)
    params_sweep.at[row_id, 'soc_comp_w'] = np.random.uniform(0, params_sweep.at[row_id, 'soc_comp_w'], size=N)
params_sweep

In [ ]:
# Check if one run finishes without errors
if run_shock:
    if not ray.is_initialized():
        ray.init(num_cpus=num_cpus)
    object_ref = run_model.remote(params_sweep.iloc[0])
    result = ray.get(object_ref)
    print(result)

In [ ]:
if run_shock:
    runs = [row for i, row in params_sweep.iterrows()]
    obj_ids = [run_model.remote(run) for run in runs]
    output_results = [ray.get(i) for i in tqdm(obj_ids)]
    params_sweep[output_cols] = output_results
    filename = 'data/parsweep-data-p.json'
    params_sweep.to_json(filename)

In [ ]:
if run_shock:
    ray.shutdown()
    assert not ray.is_initialized()

In [ ]:
filename = 'data/parsweep-data-p.json'
params_sweep = pd.read_json(filename)
params_sweep['Comparisons'] = params_sweep['homophily'].replace((-1, 2), ('Random', 'Segregated'))
params_sweep.rename(rename, inplace=True, axis=1)
params_sweep['Inequality'] = params_sweep['Inequality'].replace((10, 1), ('Unequal', 'Uniform'))
params_sweep['Society'] = params_sweep['Comparisons'] + ' and Growth Rate = ' + params_sweep['Growth Rate ($g$)'].astype(str)

In [ ]:
import warnings
warnings.simplefilter('ignore')

plot_data = params_sweep
plot_data['$\\alpha_{max}$'] = plot_data['$\\alpha_{max}$'].round(2)
plot_data = plot_data[plot_data['$\\delta_{max}$']==2]
plot_data = plot_data[plot_data['Growth Rate ($g$)']==0.002]
plot_data['Inequality'] = plot_data['Inequality'].round(2)


for output in np.array(output_cols)[[1, 5, 17]]: 
    g = sns.relplot(
            data=plot_data,
            x='p', 
            y=output,
            markers=True,
            legend='full',
            hue='Inequality', # 'Soc. Cap.',
            style='Comparisons', # 'Rewiring',
            col='$\\alpha_{max}$', #'$\\alpha_{max}$',
            # col='Growth Rate ($g$)',
#             col_wrap=4,
            row='$w_{max}$',
#             ci=None,
            errorbar=('pi',90),
            palette=sns.color_palette('tab10')[4:],
            facet_kws={'sharey':True, 'sharex':True},
            height=3.5, aspect=1, kind="line"
    )

    # g.fig.legend(ncol=2, bbox_to_anchor=(0.85, 1.05))
    g.fig.legend(ncol=7, bbox_to_anchor=(0.5, 1.0), loc='center', fontsize='small')
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    g._legend.remove() # Remove the other legend
    # g.set_titles(col_template="{col_name}")#, row_template="{row_name}")
    #     _ = plt.suptitle(f'{y} (Reps={reps})', size=20)

    for j, ax in enumerate(g.axes.flatten()):
        ax.grid(True)
        ax.annotate(alc[j] + ')', xy=(0.01,0.90), xycoords='axes fraction', fontsize=20)
#     plot_path = '../plots/wellbeing-abm/'
#     plt.savefig(plot_path + f'parsweep-{y}-new-soccompw.pdf')
    plt.tight_layout()
    plt.show()

# **Local sensitivity analysis**

In [ ]:
# Create parameter combinations
params = default_params.copy()
homophily = [-1, 2]
quantile = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1] # #
sigma = [0.1, 0.8]
w_upper = [0.5, 1]
wb_to_cap = [2]
alpha_upper = [0.10, 0.30, 0.50]

permut = []
cols = ['homophily', 'quantile', 'sigma', 'alpha', 'soc_comp_w', 'wb_to_cap']

for rep in range(reps):
    permut += list(product(homophily, quantile, sigma, alpha_upper, w_upper, wb_to_cap))
params_sweep = pd.DataFrame(permut, columns=cols)

for param, value in params.items():

    if param not in cols:
        if isinstance(value, np.ndarray):
            params_sweep[param] = [value]*len(params_sweep)
        else:
            params_sweep[param] = value

# Every run should have new random values
params_sweep['w_upper'] = params_sweep['soc_comp_w']
params_sweep['fb_upper'] = params_sweep['wb_to_cap']
params_sweep['alpha_upper'] = params_sweep['alpha']
params_sweep['alpha'] = params_sweep['alpha'].astype(object)
params_sweep['wb_to_cap'] = params_sweep['wb_to_cap'].astype(object)
params_sweep['soc_comp_w'] = params_sweep['soc_comp_w'].astype(object)
for row_id in range(params_sweep.shape[0]):
    params_sweep.at[row_id, 'alpha'] = np.random.uniform(0, params_sweep.at[row_id, 'alpha'], size=N)
    params_sweep.at[row_id, 'wb_to_cap'] = np.random.uniform(0, params_sweep.at[row_id, 'wb_to_cap'], size=N)
    params_sweep.at[row_id, 'soc_comp_w'] = np.random.uniform(0, params_sweep.at[row_id, 'soc_comp_w'], size=N)
params_sweep

In [ ]:
# Check if one run finishes without errors
if run_lsa:
    if not ray.is_initialized():
        ray.init(num_cpus=num_cpus)
    object_ref = run_model.remote(params_sweep.iloc[0])
    result = ray.get(object_ref)
    print(result)

In [ ]:
if run_lsa:
    runs = [row for i, row in params_sweep.iterrows()]
    obj_ids = [run_model.remote(run) for run in runs]
    output_results = [ray.get(i) for i in tqdm(obj_ids)]
    params_sweep[output_cols] = output_results
    filename = 'data/parsweep-data-lsa.json'
    params_sweep.to_json(filename)

In [ ]:
if run_lsa:
    ray.shutdown()
    assert not ray.is_initialized()

In [ ]:
filename = 'data/parsweep-data-lsa.json'
params_sweep = pd.read_json(filename)
params_sweep['Comparisons'] = params_sweep['homophily'].replace((-1, 2), ('Random', 'Segregated'))
params_sweep['homophily'].replace({-1:0},inplace=True)
params_sweep.rename(rename, inplace=True, axis=1)

In [ ]:
import warnings
warnings.simplefilter('ignore')

plot_data = params_sweep
plot_data['$\\alpha_{max}$'] = plot_data['$\\alpha_{max}$'].round(2)
plot_data = plot_data[plot_data['$\\delta_{max}$']==2]
plot_data['Inequality'] = plot_data['Inequality'].round(2)
plot_data['$\\beta_{swb} / \\beta_{cap}$'] = plot_data['$\\frac{\\beta_{swb}}{\\beta_{cap}}$']

for output in np.array(output_cols)[[1, 5, 6, 7, 17]]: 
    g = sns.relplot(
            data=plot_data,
            x='Quantile', 
            y=output,
            markers=True,
            legend='full',
            hue='Inequality', 
            style='Comparisons', 
            col='$\\alpha_{max}$',
            row='$w_{max}$',
            errorbar=('pi',90),
            palette='tab10',
            facet_kws={'sharey':True, 'sharex':True},
            height=3.5, aspect=1.25, kind="line"
    )

    g.fig.legend(ncol=7, bbox_to_anchor=(0.53, 1.03), loc='center', fontsize='x-small', title='$\\alpha_{max}$')
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    g._legend.remove() # Remove the other legend
   
    for j, ax in enumerate(g.axes.flatten()):
        ax.grid(True)
        ax.annotate(alc[j+4] + ')', xy=(0.01,0.90), xycoords='axes fraction', fontsize=20)
#     plot_path = '../plots/wellbeing-abm/'
#     plt.savefig(plot_path + f'parsweep-{y}-new-soccompw.pdf')
    plt.tight_layout()
    plt.show()

# **Easterlin sensitivity analysis**

In [ ]:
# Create parameter combinations
params = default_params.copy()
homophily = [-1, 2]
quantile = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1] # #
sigma = [0.1, 0.8]
w_upper = [1]
growth_rate = [-0.002, 0.002, 0.004]
wb_to_cap = [2]
alpha_upper = [0, 0.30, 0.50]

permut = []
cols = ['homophily', 'quantile', 'sigma', 'growth_rate', 'alpha', 'soc_comp_w', 'wb_to_cap']

for rep in range(reps):
    permut += list(product(homophily, quantile, sigma, growth_rate, alpha_upper, w_upper, wb_to_cap))
params_sweep = pd.DataFrame(permut, columns=cols)

for param, value in params.items():

    if param not in cols:
        if isinstance(value, np.ndarray):
            params_sweep[param] = [value]*len(params_sweep)
        else:
            params_sweep[param] = value

# Every run should have new random values
params_sweep['w_upper'] = params_sweep['soc_comp_w']
params_sweep['fb_upper'] = params_sweep['wb_to_cap']
params_sweep['alpha_upper'] = params_sweep['alpha']
params_sweep['alpha'] = params_sweep['alpha'].astype(object)
params_sweep['wb_to_cap'] = params_sweep['wb_to_cap'].astype(object)
params_sweep['soc_comp_w'] = params_sweep['soc_comp_w'].astype(object)
for row_id in range(params_sweep.shape[0]):
    params_sweep.at[row_id, 'alpha'] = np.random.uniform(0, params_sweep.at[row_id, 'alpha'], size=N)
    params_sweep.at[row_id, 'wb_to_cap'] = np.random.uniform(0, params_sweep.at[row_id, 'wb_to_cap'], size=N)
    params_sweep.at[row_id, 'soc_comp_w'] = np.random.uniform(0, params_sweep.at[row_id, 'soc_comp_w'], size=N)
params_sweep

In [ ]:
if run_easterlin:
    if not ray.is_initialized():
        ray.init(num_cpus=num_cpus)
    object_ref = run_model.remote(params_sweep.iloc[0])
    result = ray.get(object_ref)
    print(result)

In [ ]:
if run_easterlin:
    runs = [row for i, row in params_sweep.iterrows()]
    obj_ids = [run_model.remote(run) for run in runs]
    output_results = [ray.get(i) for i in tqdm(obj_ids)]
    params_sweep[output_cols] = output_results
    filename = 'data/parsweep-data-lsa-easterlin.json'
    params_sweep.to_json(filename)

In [ ]:
if run_easterlin:
    ray.shutdown()
    assert not ray.is_initialized()

In [ ]:
filename = 'data/parsweep-data-lsa-easterlin.json'
params_sweep = pd.read_json(filename)
params_sweep['Comparisons'] = params_sweep['homophily'].replace((-1, 2), ('Random', 'Segregated'))
params_sweep['homophily'].replace({-1:0},inplace=True)
params_sweep.rename(rename, inplace=True, axis=1)

In [ ]:
import warnings
warnings.simplefilter('ignore')

plot_data = params_sweep
plot_data['$\\alpha_{max}$'] = plot_data['$\\alpha_{max}$'].round(2)
plot_data = plot_data[plot_data['$\\alpha_{max}$'].isin([0, 0.30, 0.50])]
plot_data = plot_data[plot_data['$\\delta_{max}$']==2]
plot_data['Inequality'] = plot_data['Inequality'].round(2)
plot_data = plot_data[plot_data['Inequality']==0.1]
plot_data['$\\beta_{swb} / \\beta_{cap}$'] = plot_data['$\\frac{\\beta_{swb}}{\\beta_{cap}}$']

for output in ['$\\beta_{swb} / \\beta_{cap}$']: 
    g = sns.relplot(
            data=plot_data,
            x='Quantile', 
            y=output,
            markers=True,
            legend='full',
            hue='$\\alpha_{max}$', 
            style='Comparisons', 
            col='Growth Rate ($g$)',
            row='Inequality',
            errorbar=('pi',90),
            palette='flare',
            facet_kws={'sharey':True, 'sharex':True},
            height=3.5, aspect=1.25, kind="line"
    )

    g.fig.legend(ncol=7, bbox_to_anchor=(0.53, 1.03), loc='center', fontsize='x-small', title='$\\alpha_{max}$')
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    g._legend.remove() # Remove the other legend

    for j, ax in enumerate(g.axes.flatten()):
        ax.grid(True)
        ax.annotate(alc[j+4] + ')', xy=(0.01,0.90), xycoords='axes fraction', fontsize=20)

    plt.tight_layout()
    plt.show()

# **Well-being traps**

In [ ]:
if run_traps:
    ray.init(num_cpus=num_cpus)

    ncols = 5
    fig, axes = plt.subplots(nrows=2, ncols=ncols, figsize=(ncols*3.5, 12), sharex=True, sharey=True)
    axes = axes.flatten()
    i = 0
    feedback = 1
    vmin, vmax = 1, -1
    perception = 'sigmoid'

    if reps > 30:
        reps = 30 # different from all the other simulations
    
    for h in [-1, 2]:
        for q in [0, 0.25, 0.50, 0.75, 1]:
    
            if h==-1: title = f'$h=0$ and $q={q}$'
            else: title = f'$h={h}$ and $q={q}$'
            
            init_swb = np.linspace(0.1, 0.9, 9)
            init_cap = np.linspace(20_000, 100_000, 9)        
            obj_ids, caps, swbs = [], [], []
            
            for cap in init_cap:
                for swb in init_swb:
                    for rep in range(reps):
                        caps += [cap]
                        swbs += [swb]
                        params = default_params.copy()
                        params.update({
                            'N':N, 'soc_to_wb':False, 'mu':10, 'sigma':0.8,
                            'growth_rate':0.002, 'homophily':h, 'avg_degree':10, 'quantile':q,
                            'soc_comp_w':np.random.uniform(0, 1, size=N), 'soc_comparison':True,
                            'slope_sig':-0.0001, 'n_steps':99, 'beta':0, 'int_size':-10_000, 'rel_size':0,
                            'comparison_kind':'quantile', 'k':3, 'p':0.5, 'perception':perception,
                            'wb_to_cap':np.random.uniform(0, 2, size=N),
                            'alpha':np.random.uniform(0, 0.5, size=N),
                        })
                        obj_ids += [run_wellbeing_traps.remote([(cap, swb, feedback), params])]
            
            output_results = [ray.get(i) for i in tqdm(obj_ids)]
            traps_results = pd.DataFrame()
            traps_results['Initial Income'] = np.array(caps).astype(int) 
            traps_results['SWB Setpoint'] = np.round(swbs,2)
            traps_results[['Final Income', 'Final SWB', 'Long-term Change']] = output_results
            traps_results['$\\Delta SWB$'] = traps_results['Final SWB'] - traps_results['SWB Setpoint']
        
            heatmap_data = traps_results.pivot_table(index='SWB Setpoint', columns='Initial Income', values='Long-term Change', aggfunc='mean')
            ax = axes[i]
            ax.set_title(title)
            ax = sns.heatmap(heatmap_data, cmap='coolwarm_r', ax=ax, annot=True, fmt='.2f', 
                             annot_kws={'size': 9}, cbar=False, center=0, vmin=-1, vmax=1)
            ax.invert_yaxis()
            if i%ncols==0:
                ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
            else:
                ax.set_ylabel(None)
    
            poss_vmin = heatmap_data.min().min()
            poss_vmax = heatmap_data.max().max()
            if poss_vmin < vmin: vmin = poss_vmin
            if poss_vmax > vmax: vmax = poss_vmax
    
            if i<ncols: ax.set_xlabel(None)
            i += 1
    
    print('Feedback: ', feedback)
    print('Perception: ', perception)
    img = ax.collections[0]  # Seaborn stores the heatmap as a collection
    cbar = fig.colorbar(img, ax=axes, orientation='horizontal', fraction=0.05, pad=0.1) 
    cbar.set_label('Long-term Change Avg.')
    mappable = cbar.mappable
    new_norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=-vmin)
    for ax in fig.axes:
        for im in ax.images:
            im.set_norm(new_norm)
        for col in ax.collections:
            col.set_norm(new_norm)
        if hasattr(ax, 'colorbar'):
            cbar = ax.colorbar
            cbar.update_normal(cbar.mappable)
    mappable.set_norm(new_norm)
    cbar.update_normal(mappable)
    plt.suptitle('Social comparison and segregation reveal well-being traps')
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.25)
    plt.show()

In [ ]:
if run_traps:
    ray.shutdown()
    assert not ray.is_initialized()

In [ ]:
display(fig)